# Laptop Price Predictor - Model Training

In this notebook, we load the raw dataset, perform model training and validation, and save the final pipeline using joblib.

In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Model definitions
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Project imports
sys.path.append('../')
from utils.preprocess import preprocess_pipeline, raw_preprocess_transformer
from utils.encoding import get_preprocessor

# Load raw dataset
raw_df = pd.read_csv('../data/laptop_data.csv', encoding='latin-1')
print(f"Loaded raw data shape: {raw_df.shape}")

Loaded raw data shape: (1303, 13)


### Data Splitting (80% Train, 20% Test)
We split the RAW dataset before training to prevent any data leakage. Preprocessing will be done inside the training loop / evaluation pipeline.

In [2]:
X = raw_df.drop(columns=['Price_euros'])
y = raw_df['Price_euros']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Train size: (1042, 12), Test size: (261, 12)


### Defining the Processing Pipeline
We wrap the data cleaning and feature engineering step inside a `FunctionTransformer` so that it fits seamlessly into scikit-learn's Pipeline, allowing direct prediction on raw data!

In [3]:
# Test the transformer on training sample to extract column configurations
X_train_processed = raw_preprocess_transformer(X_train)
cat_cols = ['Company', 'TypeName', 'CPU Brand', 'GPU Brand', 'OpSys']
num_cols = [col for col in X_train_processed.columns if col not in cat_cols]

# Get encoding preprocessor
preprocessor = get_preprocessor(categorical_cols=cat_cols, numerical_cols=num_cols)
print("Preprocessor and Transformer configured.")

Preprocessor and Transformer configured.


### Train and Compare Models
We train and compare four regressor algorithms: Linear Regression, Decision Tree, Random Forest, and Gradient Boosting.

In [4]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    # Create complete Pipeline
    pipe = Pipeline([
        ('raw_preprocess', FunctionTransformer(raw_preprocess_transformer)),
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Train pipeline directly on raw data!
    pipe.fit(X_train, y_train)
    
    # Predict on raw data!
    y_pred = pipe.predict(X_test)
    
    # Metrics calculation
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R² Score': r2
    })

results_df = pd.DataFrame(results).sort_values(by='R² Score', ascending=False)
results_df

,Model,MAE,RMSE,R² Score
3,Gradient Boosting,176.439252,289.452521,0.835049
2,Random Forest,184.046738,304.862194,0.817019
1,Decision Tree,223.812069,330.428797,0.785041
0,Linear Regression,233.686808,330.436711,0.785031


### Select the Best Model
We select the model with the highest R² score.

In [5]:
best_model_name = results_df.iloc[0]['Model']
print(f"Best model is {best_model_name} with R² score of {results_df.iloc[0]['R² Score']:.4f}")

Best model is Gradient Boosting with R² score of 0.8350


### Save the Best Pipeline
We build the final pipeline with the best performing regressor (e.g. Random Forest or Gradient Boosting) and save it under `models/laptop_price_pipeline.pkl` using joblib.

In [6]:
best_model_instance = models[best_model_name]

final_pipeline = Pipeline([
    ('raw_preprocess', FunctionTransformer(raw_preprocess_transformer)),
    ('preprocessor', preprocessor),
    ('regressor', best_model_instance)
])

# Train on whole dataset
final_pipeline.fit(X, y)

# Ensure directory exists
os.makedirs('../models', exist_ok=True)
joblib.dump(final_pipeline, '../models/laptop_price_pipeline.pkl')
print("Trained pipeline successfully saved to ../models/laptop_price_pipeline.pkl")

Trained pipeline successfully saved to ../models/laptop_price_pipeline.pkl
